# 05 — Demande résidentielle et non résidentielle Source A projetée (2035, 2050)

Bloc C, étape 2. Reproduit la logique de `exctraction of data/source_A_grid_consumption.ipynb`
(ouvert avant d'écrire ce notebook) pour projeter la demande Source A complète — résidentielle
et non résidentielle — sous la règle de projection retenue appareil par appareil (variante
**hybride** des taux de possession).

In [1]:
import os
import numpy as np
import pandas as pd

BASE = os.path.dirname(os.getcwd())
SOURCE_A_DIR = os.path.join(BASE, "exctraction of data")
SOURCE_A_NB_PATH = os.path.join(SOURCE_A_DIR, "source_A_grid_consumption.ipynb")
CSV_FINAL = os.path.join(SOURCE_A_DIR, "output", "CSV_final.csv")

SPLIT_PATH = os.path.join(os.getcwd(), "output", "split_abc_projete.csv")
POSS_PATH = os.path.join(os.getcwd(), "output", "taux_possession_projetes.csv")
DEMANDE_OUT_PATH = os.path.join(os.getcwd(), "output", "demande_source_A_projetee.csv")
C1_OUT_PATH = os.path.join(os.getcwd(), "output", "c1_avail_exterior.csv")


## Lire les constantes de 2025 sans les recopier en dur

Les consommations unitaires, le facteur de calibration résidentiel `k` et les parts
General/Industriel ne sont **pas exportés** par `source_A_grid_consumption.ipynb` (commentaire
du notebook : *"kept in memory for traceability, not exported here"*). Pour ne rien réécrire en
dur, on ré-exécute ses cellules de code telles quelles, avec son propre répertoire comme
répertoire de travail (ses chemins relatifs le supposent), et on récupère les variables qui nous
intéressent depuis l'espace de noms résultant. Sa propre sortie stdout est capturée et masquée
ici pour ne pas dupliquer tout son affichage.

In [2]:
import io
import contextlib
import nbformat as nbf_read

nb2025 = nbf_read.read(SOURCE_A_NB_PATH, as_version=4)
ns2025 = {}
_cwd = os.getcwd()
os.chdir(SOURCE_A_DIR)
_buf = io.StringIO()
try:
    with contextlib.redirect_stdout(_buf):
        for c in nb2025.cells:
            if c.cell_type == "code":
                exec(compile(c.source, "<source_A_2025>", "exec"), ns2025)
finally:
    os.chdir(_cwd)

print(f"source_A_grid_consumption.ipynb re-execute : {len(ns2025)} variables recuperees, "
      f"{len(_buf.getvalue().splitlines())} lignes de sortie masquees.")


source_A_grid_consumption.ipynb re-execute : 195 variables recuperees, 390 lignes de sortie masquees.


In [3]:
KWH_FRIDGE, KWH_AC = ns2025["KWH_FRIDGE"], ns2025["KWH_AC"]
KWH_CALETON, KWH_MICRO = ns2025["KWH_CALETON"], ns2025["KWH_MICRO"]
FRAC_ELEC_CALETON = ns2025["FRAC_ELEC_CALETON"]
KWH_INDOOR_BULB = ns2025["KWH_INDOOR_BULB"]
KWH_TV, KWH_RADIO = ns2025["KWH_TV"], ns2025["KWH_RADIO"]
KWH_PHONE, KWH_LAPTOP = ns2025["KWH_PHONE"], ns2025["KWH_LAPTOP"]
KWH_BLENDER, KWH_WASHING, KWH_ANTENNA = ns2025["KWH_BLENDER"], ns2025["KWH_WASHING"], ns2025["KWH_ANTENNA"]
kwh_fan = ns2025["kwh_fan"]
calib_df = ns2025["calib_df"]
GENERAL_SHARES = ns2025["GENERAL_SHARES"]
INDUSTRIAL_SHARES = ns2025["INDUSTRIAL_SHARES"]
census_name = ns2025["census_name"]
df_out_2025 = ns2025["df_out"]
df_eu_mwh_2025 = ns2025["df_eu_mwh"]

print("Consommations unitaires (kWh/HH/an), lues depuis 2025 :")
for name in ["KWH_FRIDGE", "KWH_AC", "KWH_CALETON", "KWH_MICRO", "KWH_INDOOR_BULB",
             "KWH_TV", "KWH_RADIO", "KWH_PHONE", "KWH_LAPTOP", "KWH_BLENDER", "KWH_WASHING", "KWH_ANTENNA"]:
    print(f"  {name:<16}: {ns2025[name]:.3f}")
print(f"  FRAC_ELEC_CALETON : {FRAC_ELEC_CALETON}")


Consommations unitaires (kWh/HH/an), lues depuis 2025 :
  KWH_FRIDGE      : 458.000
  KWH_AC          : 451.000
  KWH_CALETON     : 824.000
  KWH_MICRO       : 50.000
  KWH_INDOOR_BULB : 57.062
  KWH_TV          : 21.900
  KWH_RADIO       : 26.280
  KWH_PHONE       : 14.600
  KWH_LAPTOP      : 38.325
  KWH_BLENDER     : 3.011
  KWH_WASHING     : 87.600
  KWH_ANTENNA     : 3.893
  FRAC_ELEC_CALETON : 0.975


### Point de vérification — le mixeur a-t-il un facteur de pénétration en 2025 ?

In [4]:
elec_cell_src = next(c.source for c in nb2025.cells
                      if c.cell_type == "code" and "KWH_BLENDER" in c.source and "elec  =" in c.source)
lines = elec_cell_src.splitlines()
start = next(i for i, l in enumerate(lines) if "elec  = (" in l)
end = next(i for i, l in enumerate(lines[start:], start) if l.strip() == ")") + 1
print("\n".join(lines[start:end]))


    elec  = (
        KWH_TV      * p['tv']        * hh_grid +
        KWH_RADIO   * p['radio']     * hh_grid +
        KWH_PHONE   * p['phone']     * hh_grid +
        KWH_LAPTOP  * p['laptop']    * hh_grid +
        KWH_BLENDER * hh_grid +
        KWH_WASHING * p['lavadora']  * hh_grid +
        KWH_ANTENNA * p['antenna']   * hh_grid +
        KWH_MICRO   * p['microwave'] * hh_grid
    )


Confirmé : `KWH_BLENDER * hh_grid` n'est multiplié par aucun terme de pénétration —
100 % des ménages raccordés sont supposés équipés. Traitement conservé à l'identique pour 2035
et 2050 (`KWH_BLENDER * hh_A(m,t)`, sans facteur de possession) : le corriger changerait la
cohérence avec 2025, ce que la consigne interdit explicitement.

## Clé de jointure — noms d'affichage (AETN) vs recensement

`df_out` de 2025 porte des noms d'affichage (`El Sena`, `Santa Rosa del Abuná`...) déjà
réconciliés avec le recensement via sa propre fonction `census_name()`. On construit la table de
correspondance vers la clé `departamento + provincia + municipio` utilisée dans le reste du
bloc C.

In [5]:
raw = pd.read_csv(CSV_FINAL, encoding="utf-8")
mm = raw[raw["MUNICIPIO/TIOC"].notna() & (raw["MUNICIPIO/TIOC"].astype(str).str.strip() != "")].copy()
assert len(mm) == 21

df_out_2025 = df_out_2025.copy()
df_out_2025["municipio_census"] = df_out_2025["municipality"].map(census_name)

key_map = df_out_2025[["municipality", "department", "municipio_census"]].merge(
    mm[["DEPARTAMENTO", "PROVINCIA", "MUNICIPIO/TIOC"]],
    left_on=["department", "municipio_census"], right_on=["DEPARTAMENTO", "MUNICIPIO/TIOC"],
    how="left", indicator=True,
)
assert (key_map["_merge"] == "both").all(), key_map.loc[key_map["_merge"] != "both"]
key_map = key_map.rename(columns={"DEPARTAMENTO": "departamento", "PROVINCIA": "provincia",
                                   "MUNICIPIO/TIOC": "municipio"})[
    ["municipality", "department", "departamento", "provincia", "municipio"]]
assert len(key_map) == 21
key_map


,municipality,department,departamento,provincia,municipio
0,Riberalta,Beni,Beni,Vaca Diez,Riberalta
1,Guayaramerín,Beni,Beni,Vaca Diez,Guayaramerín
2,Reyes,Beni,Beni,General José Ballivián,Reyes
3,Santa Rosa,Beni,Beni,General José Ballivián,Santa Rosa
4,Exaltación,Beni,Beni,Yacuma,Exaltación
5,El Sena,Pando,Pando,Madre de Dios,Sena
6,Puerto Gonzalo Moreno,Pando,Pando,Madre de Dios,Puerto Gonzalo Moreno
7,Villa Nueva,Pando,Pando,Federico Román,Villa Nueva
8,Cobija,Pando,Pando,Nicolás Suárez,Cobija
9,Porvenir,Pando,Pando,Nicolás Suárez,Porvenir


## Préambule — construire la variante hybride des taux de possession

Traitement retenu appareil par appareil (preuve de l'étape 1 du notebook 04, par appareil, pas
une règle unique) :

| Traitement | Appareils |
|---|---|
| `f(accès)` (variante centrale) | réfrigérateur, climatisation, micro-ondes, machine à laver, ordinateur, internet, téléphone |
| Gelé 2024 (variante basse) | radio, télévision, calefón |

Radio et TV sont gelés parce que le test de l'étape 1 montre que le gel les prédit mieux.
Calefón est gelé parce que son R² 2024 est de 0.29 (l'accès n'explique pas sa possession).
Écrit dans `taux_possession_projetes.csv` avec `variante = 'hybride'`, sans toucher aux deux
variantes existantes.

In [6]:
HYBRIDE_CENTRALE = ["refrigerador", "aire_acondicionado", "microondas", "lavadora",
                     "computadora", "telefono", "internet"]
HYBRIDE_BASSE = ["radio", "television", "calefon"]

poss = pd.read_csv(POSS_PATH)
poss = poss[poss["variante"] != "hybride"]  # idempotent si la cellule est ré-exécutée

poss_c = poss[(poss["variante"] == "centrale") & (poss["appareil"].isin(HYBRIDE_CENTRALE))].copy()
poss_b = poss[(poss["variante"] == "basse") & (poss["appareil"].isin(HYBRIDE_BASSE))].copy()
poss_hybride = pd.concat([poss_c, poss_b], ignore_index=True)
poss_hybride["variante"] = "hybride"

assert len(poss_hybride) == 126 * (len(HYBRIDE_CENTRALE) + len(HYBRIDE_BASSE))
assert poss_hybride["possession"].between(0, 1).all()

poss_out = pd.concat([poss, poss_hybride], ignore_index=True)
poss_out.to_csv(POSS_PATH, index=False)
print(f"variante hybride ecrite : {len(poss_hybride)} lignes ajoutees a {POSS_PATH}")
print(f"fichier total : {len(poss_out)} lignes")


variante hybride ecrite : 1260 lignes ajoutees a c:\Valen\Tfe\bolivia-energy-data\projections\output\taux_possession_projetes.csv
fichier total : 3780 lignes


In [7]:
ctrl_hyb = poss_out[(poss_out["variante"] == "hybride") & (poss_out["annee"] == 2024)]
ctrl_basse = poss_out[(poss_out["variante"] == "basse") & (poss_out["annee"] == 2024)]
chk = ctrl_hyb.merge(ctrl_basse, on=["departamento", "provincia", "municipio", "appareil", "trajectoire", "annee"],
                      suffixes=("_hyb", "_basse"), how="left", indicator=True)
assert (chk["_merge"] == "both").all()
assert np.allclose(chk["possession_hyb"].values, chk["possession_basse"].values, atol=1e-9)
print("controle : en 2024, hybride == basse == observe (identite triviale par construction) -- OK")


controle : en 2024, hybride == basse == observe (identite triviale par construction) -- OK


## Étape 1 — Diagnostic de `k(m, secteur)`

Seul le secteur résidentiel (`HOUSEHOLDS`) a un `k` au sens de la consigne :
`k(m) = res_aetn_kWh(m) / bottom_up(m)` (le `calib_factor` de 2025). Les secteurs General,
Industriel, Éclairage public et Autre **n'ont pas de bottom-up indépendant** en 2025 — ce sont
directement les valeurs AETN mesurées, ventilées par parts fixes (`GENERAL_SHARES`,
`INDUSTRIAL_SHARES`). Leur `k` vaut donc trivialement 1 : rien à figer pour eux, ils reçoivent
des drivers démographiques propres à l'étape 4.

In [8]:
calib_key = calib_df.merge(key_map, on="municipality", how="left", indicator=True)
assert (calib_key["_merge"] == "both").all()
k_res_table = calib_key[["departamento", "provincia", "municipio", "hh_grid", "bu_total_kWh",
                          "res_aetn_kWh", "calib_factor"]].rename(columns={"calib_factor": "k_res"})

k = k_res_table["k_res"]
med = k.median()
print(f"k_res sur les 21 municipalites : min={k.min():.4f}  max={k.max():.4f}  mediane={med:.4f}")

outliers = k_res_table[(k > 2 * med) | (k < med / 2)]
print(f"\nMunicipalites avec k_res s'ecartant de la mediane par un facteur > 2 : {len(outliers)}")
print(outliers[["departamento", "provincia", "municipio", "k_res"]].to_string(index=False))

k_res_table.sort_values("k_res").round(4)


k_res sur les 21 municipalites : min=1.1020  max=5.6651  mediane=2.9323

Municipalites avec k_res s'ecartant de la mediane par un facteur > 2 : 1
departamento     provincia             municipio    k_res
       Pando Madre de Dios Puerto Gonzalo Moreno 1.102001


,departamento,provincia,municipio,hh_grid,bu_total_kWh,res_aetn_kWh,k_res
6,Pando,Madre de Dios,Puerto Gonzalo Moreno,1385,4.916782e+05,541830.0,1.1020
16,Pando,Abuná,Santa Rosa,568,2.362621e+05,378030.0,1.6000
12,Pando,Nicolás Suárez,Bella Flor,569,2.174351e+05,480150.0,2.2082
4,Beni,Yacuma,Exaltación,292,7.642272e+04,194340.0,2.5430
17,Pando,Federico Román,Santos Mercado,188,4.903183e+04,125120.0,2.5518
0,Beni,Vaca Diez,Riberalta,23121,1.206438e+07,31096080.0,2.5775
8,Pando,Nicolás Suárez,Cobija,14921,1.093528e+07,29903610.0,2.7346
2,Beni,General José Ballivián,Reyes,2353,1.016006e+06,2810710.0,2.7664
9,Pando,Nicolás Suárez,Porvenir,1800,9.711478e+05,2717590.0,2.7983
13,Pando,Nicolás Suárez,Bolpebra,156,4.645789e+04,131640.0,2.8335


**Puerto Gonzalo Moreno** (k ≈ 1.10, contre une médiane de 2.93) est le seul cas hors
facteur 2 — son bottom-up couvre déjà presque tout l'AETN résidentiel mesuré, signe possible
d'une pénétration d'appareils localement plus élevée que la moyenne régionale, ou d'un AETN
sous-estimé pour cette municipalité (estimation Groupe 3, cf. 2025). Ni corrigé ni retiré ici —
signalé, la décision revient à la suite.

`k` est gelé à sa valeur 2025 pour 2035 et 2050 : ce n'est pas une erreur de calibration mais une
correction structurelle permanente. Il absorbe les appareils absents du recensement (mixeur,
usages non listés), l'écart entre le taux de possession municipal et celui des seuls ménages
raccordés, et l'intensité d'usage réelle — rien de cela ne disparaît en 2035.

## Étape 2 — Assertion de continuité (BLOQUANTE)

Applique la méthode complète de 2035 aux entrées de 2024 : possession observée 2024 (variante
hybride == observée en 2024, contrôle ci-dessus), `hh_A(m,2024)` du recensement, `k` gelé,
consommations unitaires lues de 2025. Le résultat doit reproduire exactement `df_eu_mwh` (2025)
municipalité par municipalité.

In [9]:
RES_EU = ["FOOD_PRESERVATION", "SPACE_COOLING", "HEAT_LOW_T_HW", "LIGHTING_R_C", "ELECTRICITY"]

poss_wide = poss_out[poss_out["variante"] == "hybride"].pivot_table(
    index=["departamento", "provincia", "municipio", "cluster", "trajectoire", "annee"],
    columns="appareil", values="possession").reset_index()

split_abc = pd.read_csv(SPLIT_PATH)
assert len(split_abc) == 126

full = poss_wide.merge(split_abc[["departamento", "provincia", "municipio", "trajectoire", "annee", "hh_A"]],
                        on=["departamento", "provincia", "municipio", "trajectoire", "annee"],
                        how="left", indicator=True)
assert (full["_merge"] == "both").all()
full = full.drop(columns="_merge")

full = full.merge(key_map[["departamento", "provincia", "municipio", "municipality"]],
                   on=["departamento", "provincia", "municipio"], how="left", indicator=True)
assert (full["_merge"] == "both").all()
full = full.drop(columns="_merge")

full = full.merge(kwh_fan.reset_index(name="kwh_fan"),
                   on="municipality", how="left", indicator=True)
assert (full["_merge"] == "both").all()
full = full.drop(columns="_merge")

full = full.merge(k_res_table[["departamento", "provincia", "municipio", "k_res"]],
                   on=["departamento", "provincia", "municipio"], how="left", indicator=True)
assert (full["_merge"] == "both").all()
full = full.drop(columns="_merge")

print(f"scaffold residentiel : {full.shape}  (attendu ({126}, {full.shape[1]}))")
full.head()


scaffold residentiel : (126, 20)  (attendu (126, 20))


,departamento,provincia,municipio,cluster,trajectoire,annee,aire_acondicionado,calefon,computadora,internet,lavadora,microondas,radio,refrigerador,telefono,television,hh_A,municipality,kwh_fan,k_res
0,Beni,General José Ballivián,Reyes,C1,acces_2035,2024,0.052388,0.011714,0.147086,0.223867,0.179606,0.064407,0.418232,0.422804,0.838113,0.489827,2353.000000,Reyes,84.8625,2.76643
1,Beni,General José Ballivián,Reyes,C1,acces_2035,2035,0.088669,0.011714,0.256357,0.340825,0.336097,0.076686,0.418232,0.588104,0.898073,0.489827,3309.977670,Reyes,84.8625,2.76643
2,Beni,General José Ballivián,Reyes,C1,acces_2035,2050,0.088669,0.011714,0.256357,0.340825,0.336097,0.076686,0.418232,0.588104,0.898073,0.489827,3434.334051,Reyes,84.8625,2.76643
3,Beni,General José Ballivián,Reyes,C1,acces_2050,2024,0.052388,0.011714,0.147086,0.223867,0.179606,0.064407,0.418232,0.422804,0.838113,0.489827,2353.000000,Reyes,84.8625,2.76643
4,Beni,General José Ballivián,Reyes,C1,acces_2050,2035,0.066002,0.011714,0.209063,0.259797,0.259195,0.058972,0.418232,0.470577,0.862608,0.489827,2948.670519,Reyes,84.8625,2.76643


In [10]:
food = KWH_FRIDGE * full["refrigerador"] * full["hh_A"]
cool = KWH_AC * full["aire_acondicionado"] * full["hh_A"] + full["kwh_fan"] * full["hh_A"]
heat = KWH_CALETON * full["calefon"] * FRAC_ELEC_CALETON * full["hh_A"]
light = KWH_INDOOR_BULB * full["hh_A"]
elec = (KWH_TV * full["television"] * full["hh_A"]
        + KWH_RADIO * full["radio"] * full["hh_A"]
        + KWH_PHONE * full["telefono"] * full["hh_A"]
        + KWH_LAPTOP * full["computadora"] * full["hh_A"]
        + KWH_BLENDER * full["hh_A"]
        + KWH_WASHING * full["lavadora"] * full["hh_A"]
        + KWH_ANTENNA * full["radio"] * full["hh_A"]
        + KWH_MICRO * full["microondas"] * full["hh_A"])

full["FOOD_PRESERVATION"] = food * full["k_res"] / 1000
full["SPACE_COOLING"] = cool * full["k_res"] / 1000
full["HEAT_LOW_T_HW"] = heat * full["k_res"] / 1000
full["LIGHTING_R_C"] = light * full["k_res"] / 1000
full["ELECTRICITY"] = elec * full["k_res"] / 1000

assert (full[RES_EU] >= -1e-9).all().all()
print("bottom-up residentiel calcule pour 21 municipalites x 2 trajectoires x 3 annees.")


bottom-up residentiel calcule pour 21 municipalites x 2 trajectoires x 3 annees.


In [11]:
check24 = full[(full["annee"] == 2024) & (full["trajectoire"] == "acces_2035")]
cmp_res = check24.merge(df_eu_mwh_2025.reset_index(), on="municipality", suffixes=("_check", "_ref"),
                         how="left", indicator=True)
assert (cmp_res["_merge"] == "both").all()

print("Continuite residentielle, par usage final (max erreur relative sur 21 municipalites) :")
max_rel_res = 0.0
for eu in RES_EU:
    err = (cmp_res[f"{eu}_check"] - cmp_res[f"{eu}_ref"]).abs()
    rel = (err / cmp_res[f"{eu}_ref"].abs().clip(lower=1e-9)).max()
    max_rel_res = max(max_rel_res, rel)
    print(f"  {eu:<20}: {rel*100:.6f} %")

tot_check = cmp_res[[f"{eu}_check" for eu in RES_EU]].sum(axis=1)
tot_ref = cmp_res[[f"{eu}_ref" for eu in RES_EU]].sum(axis=1)
rel_tot_res = ((tot_check - tot_ref).abs() / tot_ref.abs()).max()
print(f"\nmax erreur relative, total residentiel par municipalite : {rel_tot_res*100:.6f} %")

assert rel_tot_res < 0.001, f"CONTINUITE RESIDENTIELLE ECHOUEE : {rel_tot_res*100:.4f}% >= 0.1%"
print("CONTINUITE RESIDENTIELLE : OK (< 0.1%)")


Continuite residentielle, par usage final (max erreur relative sur 21 municipalites) :
  FOOD_PRESERVATION   : 0.000000 %
  SPACE_COOLING       : 0.000000 %
  HEAT_LOW_T_HW       : 0.000000 %
  LIGHTING_R_C        : 0.000000 %


  ELECTRICITY         : 0.000000 %

max erreur relative, total residentiel par municipalite : 0.000000 %
CONTINUITE RESIDENTIELLE : OK (< 0.1%)


## Étape 4 (secteurs non résidentiels) — anticipée ici pour clôturer la continuité totale

Le total de continuité de l'étape 2 porte sur **Source A entière**, pas seulement le
résidentiel. Les secteurs General, Industriel, Éclairage public et Autre n'ont pas de bottom-up
en 2025 (valeurs AETN directement ventilées) : à `annee = 2024`, leur projection reproduit donc
trivialement la valeur observée. Les drivers 2035/2050 :

| Secteur | Driver |
|---|---|
| Éclairage public | proportionnel à `hh_A(m,t)` (cohérent avec la règle `hh // 10` du §4.3) |
| General | proportionnel à `hh_A(m,t)` |
| Industriel | gelé en niveau 2025 (aucun driver démographique : castaña, riz, bois) |
| Autre | gelé en niveau 2025 (résiduel non affecté à un driver spécifique) |

In [12]:
nonres_2024 = df_out_2025.merge(key_map[["municipality", "departamento", "provincia", "municipio"]],
                                 on="municipality", how="left", indicator=True)
assert (nonres_2024["_merge"] == "both").all()
nonres_2024 = nonres_2024[["departamento", "provincia", "municipio",
                            "general_MWh", "industrial_MWh", "public_lighting_MWh", "other_MWh"]].rename(
    columns={c: f"{c}_2024" for c in ["general_MWh", "industrial_MWh", "public_lighting_MWh", "other_MWh"]})

hhA_2024 = split_abc[split_abc["annee"] == 2024][["departamento", "provincia", "municipio", "trajectoire", "hh_A"]].rename(
    columns={"hh_A": "hh_A_2024"})

full = full.merge(nonres_2024, on=["departamento", "provincia", "municipio"], how="left", indicator=True)
assert (full["_merge"] == "both").all()
full = full.drop(columns="_merge")

full = full.merge(hhA_2024, on=["departamento", "provincia", "municipio", "trajectoire"], how="left", indicator=True)
assert (full["_merge"] == "both").all()
full = full.drop(columns="_merge")

ratio = full["hh_A"] / full["hh_A_2024"]
full["general_MWh"] = full["general_MWh_2024"] * ratio
full["public_lighting_MWh"] = full["public_lighting_MWh_2024"] * ratio
full["industrial_MWh"] = full["industrial_MWh_2024"]
full["other_MWh"] = full["other_MWh_2024"]

full["total_source_A_MWh"] = (full[RES_EU].sum(axis=1) + full["general_MWh"] + full["industrial_MWh"]
                               + full["public_lighting_MWh"] + full["other_MWh"])
assert (full["total_source_A_MWh"] >= -1e-9).all()


In [13]:
check24b = full[(full["annee"] == 2024) & (full["trajectoire"] == "acces_2035")]
cmp_tot = check24b.merge(df_out_2025[["municipality", "total_MWh"]], on="municipality", how="left", indicator=True)
assert (cmp_tot["_merge"] == "both").all()

rel_tot_all = ((cmp_tot["total_source_A_MWh"] - cmp_tot["total_MWh"]).abs() / cmp_tot["total_MWh"].abs()).max()
print(f"Continuite TOTALE (tous secteurs), max erreur relative par municipalite : {rel_tot_all*100:.6f} %")
assert rel_tot_all < 0.001, f"CONTINUITE TOTALE ECHOUEE : {rel_tot_all*100:.4f}% >= 0.1%"
print("CONTINUITE TOTALE : OK (< 0.1%)")

region_total_2024_gwh = check24b["total_source_A_MWh"].sum() / 1000
print(f"\nTotal regional reconstruit, 2024 (= reference 2025) : {region_total_2024_gwh:.4f} GWh")


Continuite TOTALE (tous secteurs), max erreur relative par municipalite : 0.001845 %
CONTINUITE TOTALE : OK (< 0.1%)

Total regional reconstruit, 2024 (= reference 2025) : 180.1031 GWh


### Écart 177,6 vs 180,10 GWh — élucidé, en lecture seule

Quatre tests, sans rien corriger :

1. Somme AETN telle que lue par `source_A_grid_consumption.ipynb` (déjà obtenue à l'étape 2
   ci-dessus).
2. Sous-ensembles de tarifs (sans `Other`, sans éclairage public...) : testés systématiquement.
3. Sous-ensembles de municipalités (une exclue à la fois) : testés systématiquement.
4. Historique git du notebook de 2025 : les sorties stockées dans chaque commit sont lues
   directement (pas de ré-exécution d'un ancien code).

In [14]:
import itertools
import re
import subprocess

CATS = ["residential_MWh", "general_MWh", "industrial_MWh", "public_lighting_MWh", "other_MWh"]
MEMOIRE_GWH = 177.6

ref_2025_gwh = df_out_2025[CATS].sum().sum() / 1000
print(f"1. Total AETN reconstruit (5 tarifs, 21 municipalites) = {ref_2025_gwh:.4f} GWh")
print(f"   Chiffre cite dans la consigne = {MEMOIRE_GWH} GWh "
      f"(ecart {abs(ref_2025_gwh - MEMOIRE_GWH) / MEMOIRE_GWH * 100:.2f} %)")

print("\n2. Sous-ensembles de tarifs les plus proches de 177.6 GWh (21 municipalites) :")
best_cat = sorted(
    (abs(df_out_2025[list(combo)].sum().sum() / 1000 - MEMOIRE_GWH), combo,
     df_out_2025[list(combo)].sum().sum() / 1000)
    for r in range(1, len(CATS) + 1) for combo in itertools.combinations(CATS, r)
)
for d, combo, v in best_cat[:3]:
    print(f"   {v:.4f} GWh (ecart {d:.4f})  {combo}")
print("   -> aucune combinaison de tarifs ne s'approche a moins de 0.7 GWh.")

print("\n3. Exclusion d'une municipalite a la fois (5 tarifs, 20 restantes) :")
muni_hits = []
for muni in df_out_2025["municipality"]:
    v = df_out_2025.loc[df_out_2025["municipality"] != muni, CATS].sum().sum() / 1000
    d = abs(v - MEMOIRE_GWH)
    if d < 0.1:
        muni_hits.append((muni, v, d))
        print(f"   en excluant {muni} : {v:.4f} GWh (ecart {d:.4f}, {d / MEMOIRE_GWH * 100:.3f} %)")

print("\n4. Historique git de source_A_grid_consumption.ipynb (sorties stockees, pas de re-execution) :")
log = subprocess.run(
    ["git", "log", "--oneline", "--follow", "--", "exctraction of data/source_A_grid_consumption.ipynb"],
    cwd=BASE, capture_output=True, text=True).stdout
commits = [l.split()[0] for l in log.strip().splitlines()]
print(f"   {len(commits)} commit(s) touchant ce fichier : {log.strip()}")
for commit in commits:
    blob = subprocess.run(
        ["git", "show", f"{commit}:exctraction of data/source_A_grid_consumption.ipynb"],
        cwd=BASE, capture_output=True, text=True, encoding="utf-8", errors="replace").stdout
    m = re.search(r"GRAND TOTAL \(21 municipalities\): ([\d.]+) MWh", blob)
    print(f"   {commit}: total stocke dans les sorties = "
          f"{m.group(1) + ' MWh' if m else 'aucune sortie stockee'}")

print("\nConclusion : le notebook a toujours calcule ~180.10 GWh, y compris au tout premier "
      "commit de ce depot -- jamais 177.6 dans l'historique git. La seule exclusion qui "
      "reproduit 177.6 a moins de 0.03% pres est San Lorenzo (Group 3 -- 'Estimation', la "
      "methode la plus fragile des trois : aucune donnee AETN individuelle, pas rattachee au "
      "systeme ENDE Cobija -- cf. source_A_grid_consumption.ipynb). San Lorenzo figure dans "
      "CSV_final.csv depuis le tout premier commit (recensement), mais son estimation de "
      "consommation AETN a vraisemblablement ete ajoutee a la reconstruction Source A apres la "
      "redaction du memoire. Signale : c'est le memoire qui devra etre mis a jour (ou pas), "
      "pas le code -- aucune correction appliquee ici.")

1. Total AETN reconstruit (5 tarifs, 21 municipalites) = 180.1031 GWh
   Chiffre cite dans la consigne = 177.6 GWh (ecart 1.41 %)

2. Sous-ensembles de tarifs les plus proches de 177.6 GWh (21 municipalites) :


   178.3450 GWh (ecart 0.7450)  ('residential_MWh', 'general_MWh', 'industrial_MWh', 'public_lighting_MWh')
   180.1031 GWh (ecart 2.5031)  ('residential_MWh', 'general_MWh', 'industrial_MWh', 'public_lighting_MWh', 'other_MWh')
   170.3936 GWh (ecart 7.2064)  ('residential_MWh', 'general_MWh', 'industrial_MWh', 'other_MWh')
   -> aucune combinaison de tarifs ne s'approche a moins de 0.7 GWh.

3. Exclusion d'une municipalite a la fois (5 tarifs, 20 restantes) :
   en excluant San Lorenzo : 177.6455 GWh (ecart 0.0455, 0.026 %)

4. Historique git de source_A_grid_consumption.ipynb (sorties stockees, pas de re-execution) :


   2 commit(s) touchant ce fichier : cfc4763 energyscope scenario reality + corrections
b9f34c0 end demande scenario reality


   cfc4763: total stocke dans les sorties = 180103.04 MWh
   b9f34c0: total stocke dans les sorties = 180103.04 MWh

Conclusion : le notebook a toujours calcule ~180.10 GWh, y compris au tout premier commit de ce depot -- jamais 177.6 dans l'historique git. La seule exclusion qui reproduit 177.6 a moins de 0.03% pres est San Lorenzo (Group 3 -- 'Estimation', la methode la plus fragile des trois : aucune donnee AETN individuelle, pas rattachee au systeme ENDE Cobija -- cf. source_A_grid_consumption.ipynb). San Lorenzo figure dans CSV_final.csv depuis le tout premier commit (recensement), mais son estimation de consommation AETN a vraisemblablement ete ajoutee a la reconstruction Source A apres la redaction du memoire. Signale : c'est le memoire qui devra etre mis a jour (ou pas), pas le code -- aucune correction appliquee ici.


## Étape 4 — Tarifs AETN réellement présents

In [15]:
import re

with open(os.path.join(SOURCE_A_DIR, "data", "electricity_beni_pando_aetn_2024.csv"),
          encoding="utf-8-sig") as f:
    raw_aetn_text = f.read()

categories_found = sorted(set(re.findall(r"([A-Za-z ]+):\s*(?:[\d.,\- ]|$)", raw_aetn_text)))
anticipated = {"Residential", "General", "Industrial", "Public Lighting", "Other"}
print(f"Categories trouvees dans le texte brut AETN : {categories_found}")

mining_values = re.findall(r"Mining:\s*([^<\";]*)", raw_aetn_text)
mining_nonzero = [v.strip() for v in mining_values if v.strip() not in ("", "-", "0,00")]
print(f"\n'Mining' present dans le schema AETN : {len(mining_values)} occurrences, "
      f"valeurs non nulles/non vides : {mining_nonzero if mining_nonzero else 'aucune'}")
print("-> categorie presente dans le schema mais MWh = 0 pour toutes les lignes utilisees par les "
      "21 municipalites : aucun driver necessaire, non ventilee par source_A_grid_consumption.ipynb "
      "(cats_from_bd() ne l'extrait pas -- sans consequence puisqu'elle est toujours nulle ici).")


Categories trouvees dans le texte brut AETN : ['General', 'Industrial', 'Mining', 'Other', 'Public Lighting', 'Residential']

'Mining' present dans le schema AETN : 38 occurrences, valeurs non nulles/non vides : ['11']
-> categorie presente dans le schema mais MWh = 0 pour toutes les lignes utilisees par les 21 municipalites : aucun driver necessaire, non ventilee par source_A_grid_consumption.ipynb (cats_from_bd() ne l'extrait pas -- sans consequence puisqu'elle est toujours nulle ici).


## Étape 5 — Mapping vers les usages finaux EnergyScope

Repris à l'identique de `SECTOR_DEFS` (2025) : parts `GENERAL_SHARES`/`INDUSTRIAL_SHARES` lues
depuis le notebook de 2025, jamais recopiées en dur. Les secteurs ne sont jamais sommés entre
eux.

In [16]:
for eu, share in GENERAL_SHARES.items():
    full[f"SERVICES__{eu}"] = full["general_MWh"] * share
for eu, share in INDUSTRIAL_SHARES.items():
    full[f"INDUSTRY__{eu}"] = full["industrial_MWh"] * share
full["PUBLIC_LIGHTING__LIGHTING_P"] = full["public_lighting_MWh"]
full["SERVICES_OTHER__ELECTRICITY"] = full["other_MWh"]
for eu in RES_EU:
    full[f"HOUSEHOLDS__{eu}"] = full[eu]

usage_cols = [c for c in full.columns if "__" in c]
print(f"{len(usage_cols)} combinaisons (secteur, usage_final) :")
for c in usage_cols:
    print(f"  {c}")


16 combinaisons (secteur, usage_final) :
  SERVICES__ELECTRICITY
  SERVICES__SPACE_COOLING
  SERVICES__FOOD_PRESERVATION
  SERVICES__COOKING
  SERVICES__MECHANICAL_ENERGY_COMM
  SERVICES__LIGHTING_R_C
  SERVICES__HEAT_LOW_T_HW
  INDUSTRY__MECHANICAL_ENERGY_IND
  INDUSTRY__LIGHTING_R_C
  PUBLIC_LIGHTING__LIGHTING_P
  SERVICES_OTHER__ELECTRICITY
  HOUSEHOLDS__FOOD_PRESERVATION
  HOUSEHOLDS__SPACE_COOLING
  HOUSEHOLDS__HEAT_LOW_T_HW
  HOUSEHOLDS__LIGHTING_R_C
  HOUSEHOLDS__ELECTRICITY


In [17]:
long = full.melt(
    id_vars=["departamento", "provincia", "municipio", "cluster", "trajectoire", "annee"],
    value_vars=usage_cols, var_name="sect_eu", value_name="demande_MWh")
long[["secteur", "usage_final"]] = long["sect_eu"].str.split("__", expand=True)
long = long.drop(columns="sect_eu")
long["demande_GWh"] = long["demande_MWh"] / 1000
long = long.drop(columns="demande_MWh")

assert (long["demande_GWh"] >= -1e-9).all()
assert long.isna().sum().sum() == 0
print(f"table longue : {long.shape}")
long.head()


table longue : (2016, 9)


,departamento,provincia,municipio,cluster,trajectoire,annee,secteur,usage_final,demande_GWh
0,Beni,General José Ballivián,Reyes,C1,acces_2035,2024,SERVICES,ELECTRICITY,0.634374
1,Beni,General José Ballivián,Reyes,C1,acces_2035,2035,SERVICES,ELECTRICITY,0.892377
2,Beni,General José Ballivián,Reyes,C1,acces_2035,2050,SERVICES,ELECTRICITY,0.925904
3,Beni,General José Ballivián,Reyes,C1,acces_2050,2024,SERVICES,ELECTRICITY,0.634374
4,Beni,General José Ballivián,Reyes,C1,acces_2050,2035,SERVICES,ELECTRICITY,0.794968


## Étape 6 — `C1.ELECTRICITY.avail_exterior`

Municipalités C1 raccordées au SIN, clé de 2025 : Reyes, Santa Rosa (Beni) et Ixiamas à 100 %,
Exaltación à 50 %. Clé complète `departamento + provincia + municipio` (Santa Rosa existe aussi
en cluster C4/Pando).

In [18]:
C1_SIN_SHARES = {
    ("Beni", "General José Ballivián", "Reyes"): 1.0,
    ("Beni", "General José Ballivián", "Santa Rosa"): 1.0,
    ("La Paz", "Abel Iturralde", "Ixiamas"): 1.0,
    ("Beni", "Yacuma", "Exaltación"): 0.5,
}
sin_df = pd.DataFrame([{"departamento": d, "provincia": p, "municipio": m, "share": s}
                        for (d, p, m), s in C1_SIN_SHARES.items()])

cluster_check = split_abc.merge(sin_df, on=["departamento", "provincia", "municipio"], how="inner")[
    ["departamento", "provincia", "municipio", "cluster"]].drop_duplicates()
assert (cluster_check["cluster"] == "C1").all(), "les 4 municipalites SIN devraient toutes etre C1"
print("verification cluster : les 4 municipalites raccordees au SIN sont bien C1")
cluster_check


verification cluster : les 4 municipalites raccordees au SIN sont bien C1


,departamento,provincia,municipio,cluster
0,La Paz,Abel Iturralde,Ixiamas,C1
3,Beni,General José Ballivián,Reyes,C1
6,Beni,General José Ballivián,Santa Rosa,C1
9,Beni,Yacuma,Exaltación,C1


In [19]:
tot_by_muni = full.groupby(["departamento", "provincia", "municipio", "trajectoire", "annee"])[
    "total_source_A_MWh"].sum().reset_index()

avail = tot_by_muni.merge(sin_df, on=["departamento", "provincia", "municipio"], how="inner", indicator=True)
assert (avail["_merge"] == "both").all()
assert len(avail) == 4 * 2 * 3, len(avail)
avail["weighted_MWh"] = avail["total_source_A_MWh"] * avail["share"]

avail_out = avail.groupby(["trajectoire", "annee"])["weighted_MWh"].sum().div(1000).rename("avail_exterior_GWh").reset_index()
avail_out


,trajectoire,annee,avail_exterior_GWh
0,acces_2035,2024,11.592630
1,acces_2035,2035,25.434249
2,acces_2035,2050,29.028436
3,acces_2050,2024,11.592630
4,acces_2050,2035,18.559095
5,acces_2050,2050,29.028436


In [20]:
ctrl_2025 = avail_out[(avail_out["annee"] == 2024) & (avail_out["trajectoire"] == "acces_2035")][
    "avail_exterior_GWh"].iloc[0]
TARGET_AVAIL = 11.593
ecart_avail = abs(ctrl_2025 - TARGET_AVAIL) / TARGET_AVAIL * 100
print(f"Controle 2025 : avail_exterior reconstruit = {ctrl_2025:.4f} GWh/an, cible = {TARGET_AVAIL} GWh/an, "
      f"ecart = {ecart_avail:.3f} %")
assert ecart_avail < 1.0, f"avail_exterior 2025 s'ecarte de {ecart_avail:.2f}% (>= 1%) -- signale, pas de correction inventee"
print("OK : ecart < 1%")


Controle 2025 : avail_exterior reconstruit = 11.5926 GWh/an, cible = 11.593 GWh/an, ecart = 0.003 %
OK : ecart < 1%


## Étape 7 — Sorties

In [21]:
OUT_COLS = ["departamento", "provincia", "municipio", "cluster", "trajectoire", "annee",
            "secteur", "usage_final", "demande_GWh"]
long_out = long[OUT_COLS].copy()

assert long_out.isna().sum().sum() == 0
os.makedirs(os.path.dirname(DEMANDE_OUT_PATH), exist_ok=True)
long_out.to_csv(DEMANDE_OUT_PATH, index=False)
print(f"ecrit : {DEMANDE_OUT_PATH}  ({len(long_out)} lignes, {len(OUT_COLS)} colonnes)")

avail_out_cols = avail_out[["trajectoire", "annee", "avail_exterior_GWh"]]
assert avail_out_cols.isna().sum().sum() == 0
avail_out_cols.to_csv(C1_OUT_PATH, index=False)
print(f"ecrit : {C1_OUT_PATH}  ({len(avail_out_cols)} lignes)")


ecrit : c:\Valen\Tfe\bolivia-energy-data\projections\output\demande_source_A_projetee.csv  (2016 lignes, 9 colonnes)
ecrit : c:\Valen\Tfe\bolivia-energy-data\projections\output\c1_avail_exterior.csv  (6 lignes)


### Assertions bloquantes

In [22]:
n_munis = long_out.groupby(["departamento", "provincia", "municipio"]).ngroups
assert n_munis == 21, n_munis
print(f"1. municipalites distinctes : {n_munis} -- OK")

print("2. continuite 2024 : voir etape 2 ci-dessus -- OK (< 0.1%)")

print(f"3. avail_exterior 2025 : {ecart_avail:.3f}% d'ecart -- OK (< 1%)")

assert long_out.isna().sum().sum() == 0 and avail_out_cols.isna().sum().sum() == 0
print("4. aucun NaN dans les deux fichiers de sortie -- OK")

print("5. merges avec split_abc_projete.csv verifies avec indicator=True a chaque etape -- OK")

cluster_sum = long_out.groupby(["trajectoire", "annee"])["demande_GWh"].sum()
by_cluster = long_out.groupby(["cluster", "trajectoire", "annee"])["demande_GWh"].sum().groupby(["trajectoire", "annee"]).sum()
assert np.allclose(by_cluster.values, cluster_sum.reindex(by_cluster.index).values)
print("6. somme des clusters == total regional -- OK")

assert (long_out["demande_GWh"] >= -1e-9).all()
print("7. aucune demande negative -- OK")

print("8. aucune conversion Layers_in_out appliquee -- ce notebook ne lit jamais Layers_in_out.csv, "
      "toutes les valeurs restent en energie finale (MWh/GWh AETN) -- OK")


1. municipalites distinctes : 21 -- OK
2. continuite 2024 : voir etape 2 ci-dessus -- OK (< 0.1%)
3. avail_exterior 2025 : 0.003% d'ecart -- OK (< 1%)
4. aucun NaN dans les deux fichiers de sortie -- OK
5. merges avec split_abc_projete.csv verifies avec indicator=True a chaque etape -- OK
6. somme des clusters == total regional -- OK
7. aucune demande negative -- OK
8. aucune conversion Layers_in_out appliquee -- ce notebook ne lit jamais Layers_in_out.csv, toutes les valeurs restent en energie finale (MWh/GWh AETN) -- OK


### Demande totale par cluster et usage final

In [23]:
cluster_grp = long_out.groupby(["cluster", "usage_final", "trajectoire", "annee"])["demande_GWh"].sum()
region_grp = pd.concat({"REGION": long_out.groupby(["usage_final", "trajectoire", "annee"])["demande_GWh"].sum()},
                        names=["cluster"])
cluster_table = pd.concat([cluster_grp, region_grp]).unstack("annee").round(3)
cluster_table

annee                                         2024    2035    2050
cluster usage_final            trajectoire                        
C1      COOKING                acces_2035    0.102   0.189   0.217
                               acces_2050    0.102   0.155   0.217
        ELECTRICITY            acces_2035    2.883   6.080   6.953
                               acces_2050    2.883   4.644   6.953
        FOOD_PRESERVATION      acces_2035    3.267   9.866  11.358
...                                            ...     ...     ...
REGION  MECHANICAL_ENERGY_COMM acces_2050    1.482   2.001   2.568
        MECHANICAL_ENERGY_IND  acces_2035   12.963  12.963  12.963
                               acces_2050   12.963  12.963  12.963
        SPACE_COOLING          acces_2035   32.593  49.981  59.117
                               acces_2050   32.593  43.397  59.117

[108 rows x 3 columns]

### Intensité résidentielle par ménage Source A (kWh/ménage/an)

Demande résidentielle uniquement (les usages non résidentiels ne sont pas portés par `hh_A`) —
référence 2025 ≈ 1 577 kWh/ménage.

In [24]:
full["res_total_MWh"] = full[RES_EU].sum(axis=1)
full["intensity_kWh_hh"] = full["res_total_MWh"] * 1000 / full["hh_A"]

intens = full[full["trajectoire"] == "acces_2035"].pivot_table(
    index="municipio", columns="annee", values="intensity_kWh_hh")

region_intensity = full.groupby(["trajectoire", "annee"]).apply(
    lambda g: g["res_total_MWh"].sum() * 1000 / g["hh_A"].sum(), include_groups=False
).rename("intensity_kWh_hh_regional")
print("Intensite residentielle regionale moyenne (ponderee par hh_A), par trajectoire et annee "
      "(reference 2024 = 1577.3 kWh/hh) :")
print(region_intensity.round(1))

print("\nPar municipalite (trajectoire acces_2035), tri croissant sur 2024 :")
intens.sort_values(2024).round(1)

Intensite residentielle regionale moyenne (ponderee par hh_A), par trajectoire et annee (reference 2024 = 1577.3 kWh/hh) :
trajectoire  annee
acces_2035   2024     1577.3
             2035     1718.2
             2050     1721.1
acces_2050   2024     1577.3
             2035     1570.5
             2050     1721.1
Name: intensity_kWh_hh_regional, dtype: float64

Par municipalite (trajectoire acces_2035), tri croissant sur 2024 :


annee,2024,2035,2050
municipio,,,
Puerto Gonzalo Moreno,391.2,626.2,626.2
San Pedro,665.0,1871.0,1871.0
Nueva Esperanza,665.5,1696.3,1696.3
Santos Mercado,665.5,1419.1,1419.1
Exaltación,665.5,1434.3,1434.3
Ingavi,665.6,1635.8,1635.8
Bolpebra,843.8,1632.7,1632.7
Bella Flor,843.8,1235.3,1235.3
Villa Nueva,1019.1,1982.8,1982.8


In [25]:
extremes_2050 = intens[2050].sort_values()
print("Trois municipalites extremes en 2050 (acces_2035) :")
print(f"  min : {extremes_2050.index[0]}  ({extremes_2050.iloc[0]:.1f} kWh/hh)")
print(f"  max : {extremes_2050.index[-1]}  ({extremes_2050.iloc[-1]:.1f} kWh/hh)")
print(f"  2e plus bas : {extremes_2050.index[1]}  ({extremes_2050.iloc[1]:.1f} kWh/hh)")


Trois municipalites extremes en 2050 (acces_2035) :
  min : Puerto Gonzalo Moreno  (626.2 kWh/hh)
  max : Sena  (3198.1 kWh/hh)
  2e plus bas : Bella Flor  (1235.3 kWh/hh)


### Encadrement de la demande régionale 2050

Deux variantes extrêmes du fichier de possession appliquées à **tous** les appareils
résidentiels (les secteurs non résidentiels ne dépendent pas du fichier de possession, donc
restent identiques dans les trois cas).

In [26]:
ALL_APPAREILS = HYBRIDE_CENTRALE + HYBRIDE_BASSE


def build_residential_total(variante):
    p = poss_out[(poss_out["variante"] == variante) & (poss_out["appareil"].isin(ALL_APPAREILS))]
    pw = p.pivot_table(index=["departamento", "provincia", "municipio", "trajectoire", "annee"],
                        columns="appareil", values="possession").reset_index()
    m = pw.merge(split_abc[["departamento", "provincia", "municipio", "trajectoire", "annee", "hh_A"]],
                 on=["departamento", "provincia", "municipio", "trajectoire", "annee"], how="left", indicator=True)
    assert (m["_merge"] == "both").all()
    m = m.drop(columns="_merge")
    m = m.merge(key_map[["departamento", "provincia", "municipio", "municipality"]],
                on=["departamento", "provincia", "municipio"], how="left", indicator=True)
    assert (m["_merge"] == "both").all()
    m = m.drop(columns="_merge")
    m = m.merge(kwh_fan.reset_index(name="kwh_fan"),
                on="municipality", how="left", indicator=True)
    assert (m["_merge"] == "both").all()
    m = m.drop(columns="_merge")
    m = m.merge(k_res_table[["departamento", "provincia", "municipio", "k_res"]],
                on=["departamento", "provincia", "municipio"], how="left", indicator=True)
    assert (m["_merge"] == "both").all()

    food = KWH_FRIDGE * m["refrigerador"] * m["hh_A"]
    cool = KWH_AC * m["aire_acondicionado"] * m["hh_A"] + m["kwh_fan"] * m["hh_A"]
    heat = KWH_CALETON * m["calefon"] * FRAC_ELEC_CALETON * m["hh_A"]
    light = KWH_INDOOR_BULB * m["hh_A"]
    elec = (KWH_TV * m["television"] * m["hh_A"] + KWH_RADIO * m["radio"] * m["hh_A"]
            + KWH_PHONE * m["telefono"] * m["hh_A"] + KWH_LAPTOP * m["computadora"] * m["hh_A"]
            + KWH_BLENDER * m["hh_A"] + KWH_WASHING * m["lavadora"] * m["hh_A"]
            + KWH_ANTENNA * m["radio"] * m["hh_A"] + KWH_MICRO * m["microondas"] * m["hh_A"])
    m["res_total_MWh"] = (food + cool + heat + light + elec) * m["k_res"] / 1000
    return m


nonres_2050 = full[full["annee"] == 2050][["departamento", "provincia", "municipio", "trajectoire",
                                            "general_MWh", "industrial_MWh", "public_lighting_MWh", "other_MWh"]]
nonres_2050_sum = nonres_2050.set_index(["departamento", "provincia", "municipio", "trajectoire"])[
    ["general_MWh", "industrial_MWh", "public_lighting_MWh", "other_MWh"]].sum(axis=1)

bornes = {}
for label in ["basse", "centrale"]:
    m2050 = build_residential_total(label)
    m2050 = m2050[m2050["annee"] == 2050]
    res_idx = m2050.set_index(["departamento", "provincia", "municipio", "trajectoire"])["res_total_MWh"]
    combined = (res_idx + nonres_2050_sum.reindex(res_idx.index)).xs("acces_2035", level="trajectoire")
    bornes[label] = combined.sum() / 1000

hybride_2050 = long_out[(long_out["annee"] == 2050) & (long_out["trajectoire"] == "acces_2035")]["demande_GWh"].sum()

print("Encadrement de la demande regionale totale, 2050 :")
print(f"  borne basse (tout gele)      : {bornes['basse']:.2f} GWh")
print(f"  hybride (retenue)            : {hybride_2050:.2f} GWh")
print(f"  borne haute (tout f(acces))  : {bornes['centrale']:.2f} GWh")


Encadrement de la demande regionale totale, 2050 :
  borne basse (tout gele)      : 302.09 GWh
  hybride (retenue)            : 325.00 GWh
  borne haute (tout f(acces))  : 326.43 GWh


## Récapitulatif à rapporter

In [27]:
print("Demande regionale totale (GWh), par trajectoire et annee :")
print(long_out.groupby(["trajectoire", "annee"])["demande_GWh"].sum().round(2))

print("\navail_exterior (GWh/an) :")
print(avail_out_cols.round(4))


Demande regionale totale (GWh), par trajectoire et annee :
trajectoire  annee
acces_2035   2024     180.10
             2035     277.16
             2050     325.00
acces_2050   2024     180.10
             2035     240.38
             2050     325.00
Name: demande_GWh, dtype: float64

avail_exterior (GWh/an) :
  trajectoire  annee  avail_exterior_GWh
0  acces_2035   2024             11.5926
1  acces_2035   2035             25.4342
2  acces_2035   2050             29.0284
3  acces_2050   2024             11.5926
4  acces_2050   2035             18.5591
5  acces_2050   2050             29.0284


## Point de vigilance — `k(m)` et intensité par ménage (non corrigé)

`k` va de 1,10 à 5,66 (facteur 5) entre municipalités. Gelé, il produit un écart d'intensité
important en 2050. Table complète, triée par `k` croissant — documente la limitation, ne la
traite pas : corriger `k` casserait la continuité avec 2025 (verrouillée).

In [28]:
intens_24_50 = full[full["trajectoire"] == "acces_2035"].pivot_table(
    index=["departamento", "provincia", "municipio"], columns="annee", values="intensity_kWh_hh")[[2024, 2050]]
intens_24_50.columns = ["intensity_2024_kWh_hh", "intensity_2050_kWh_hh"]

k_full = k_res_table[["departamento", "provincia", "municipio", "k_res"]].merge(
    intens_24_50.reset_index(), on=["departamento", "provincia", "municipio"], how="left", indicator=True)
assert (k_full["_merge"] == "both").all()
k_full = k_full.drop(columns="_merge").sort_values("k_res").reset_index(drop=True)

print(f"k(m) : facteur {k_full['k_res'].max() / k_full['k_res'].min():.2f} entre min et max "
      f"({k_full['k_res'].min():.2f} a {k_full['k_res'].max():.2f})")
k_full.round(1)

k(m) : facteur 5.14 entre min et max (1.10 a 5.67)


,departamento,provincia,municipio,k_res,intensity_2024_kWh_hh,intensity_2050_kWh_hh
0,Pando,Madre de Dios,Puerto Gonzalo Moreno,1.1,391.2,626.2
1,Pando,Abuná,Santa Rosa,1.6,665.5,902.9
2,Pando,Nicolás Suárez,Bella Flor,2.2,843.8,1235.3
3,Beni,Yacuma,Exaltación,2.5,665.5,1434.3
4,Pando,Federico Román,Santos Mercado,2.6,665.5,1419.1
5,Beni,Vaca Diez,Riberalta,2.6,1344.9,1502.1
6,Pando,Nicolás Suárez,Cobija,2.7,2004.1,1591.5
7,Beni,General José Ballivián,Reyes,2.8,1194.5,1502.9
8,Pando,Nicolás Suárez,Porvenir,2.8,1509.8,1627.8
9,Pando,Nicolás Suárez,Bolpebra,2.8,843.8,1632.7


## Conclusion

Le ré-échelonnage de la trajectoire tendancielle (bloc B, notebook 03) a été corrigé : `phi`
normalise désormais la fermeture du gap pour atteindre exactement 1.0 en 2050 par construction
(plus de forçage littéral), ce qui déplace le split A/B/C de `acces_2050` en 2035 (plus de
municipalités déjà proches de l'accès complet à cette date) sans toucher 2024 ni 2050. La chaîne
2035/2050 de ce notebook, ré-exécutée avec le split corrigé, reproduit toujours exactement
(< 0.002 %) la demande résidentielle et totale de 2025, et `avail_exterior` 2025 reproduit
11.593 GWh/an à 0.003 % près — la correction du bloc B ne casse rien en aval.

L'écart 177,6 vs 180,10 GWh est élucidé (ci-dessus) : San Lorenzo, une municipalité "Groupe 3 —
Estimation" (la méthode la plus fragile de la reconstruction AETN), explique l'écart à moins de
0.03 % près une fois exclue. Le notebook de 2025 a toujours calculé ~180,10 GWh dans historique
git de ce dépôt — c'est le mémoire qui devra trancher, pas le code.

Puerto Gonzalo Moreno reste le point de vigilance documenté en fin de notebook : `k` hors norme
(facteur ~5 avec Sena) et intensité par ménage la plus basse de la région, gelés tels quels par
décision — ce n'est pas une erreur à corriger ici.